<a href="https://colab.research.google.com/github/davidriveraarbelaez/IST-Optativa_III/blob/main/Optativa_III_M%C3%B3dulo_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Laboratorio**: Construyendo un agente inteligente en Python

## Módulo 1 — Fundamentos de Agentes Inteligentes

En este laboratorio construiremos progresivamente un agente autónomo
capaz de:

- Percibir su entorno
- Tomar decisiones
- Ejecutar acciones
- Evaluar su desempeño
- Mantener memoria
- Modificar su comportamiento

### Ruta de construcción

Percepción --> Estado interno --> Decisión --> Acción --> Nuevo estado --> Evaluación -->  Aprendizaje

In [1]:
import random
import matplotlib.pyplot as plt
from dataclasses import dataclass

In [2]:
class Entorno:
    def __init__(self, filas=5, columnas=5):
        self.filas = filas
        self.columnas = columnas

        self.robot = [0, 0]

        self.suciedad = set()
        self.obstaculos = set()

        self._crear_entorno()

    def _crear_entorno(self):
        # Crear suciedad aleatoria
        while len(self.suciedad) < 6:
            posicion = (
                random.randint(0, self.filas - 1),
                random.randint(0, self.columnas - 1)
            )

            if posicion != tuple(self.robot):
                self.suciedad.add(posicion)

        # Crear obstáculos
        while len(self.obstaculos) < 4:
            posicion = (
                random.randint(0, self.filas - 1),
                random.randint(0, self.columnas - 1)
            )

            if (
                posicion != tuple(self.robot)
                and posicion not in self.suciedad
            ):
                self.obstaculos.add(posicion)

    def mostrar(self):
        matriz = []

        for i in range(self.filas):
            fila = []

            for j in range(self.columnas):

                posicion = (i, j)

                if posicion == tuple(self.robot):
                    fila.append("🤖")

                elif posicion in self.obstaculos:
                    fila.append("⬛")

                elif posicion in self.suciedad:
                    fila.append("🟫")

                else:
                    fila.append("⬜")

            matriz.append(fila)

        for fila in matriz:
            print(" ".join(fila))

In [3]:
entorno = Entorno()
entorno.mostrar()

🤖 ⬛ 🟫 ⬜ ⬛
⬜ ⬜ ⬜ 🟫 🟫
⬜ ⬜ 🟫 ⬛ 🟫
⬜ ⬜ ⬜ ⬜ ⬜
⬜ ⬜ ⬛ 🟫 ⬜


**Ciclo percepción - acción**

In [4]:
class Agente:

    def __init__(self, entorno):
        self.entorno = entorno

    def percibir(self):
        posicion = tuple(self.entorno.robot)

        return {
            "posicion": posicion,
            "sucio": posicion in self.entorno.suciedad,
            "obstaculo": posicion in self.entorno.obstaculos
        }

    def actuar(self, accion):
        print(f"Acción ejecutada: {accion}")

In [5]:
agente = Agente(entorno)

percepcion = agente.percibir()

print(percepcion)

{'posicion': (0, 0), 'sucio': False, 'obstaculo': False}


**Agente reactivo**

In [6]:
class AgenteReactivo(Agente):

    def decidir(self, percepcion):

        if percepcion["sucio"]:
            return "ASPIRAR"

        if percepcion["obstaculo"]:
            return "CAMBIAR_DIRECCION"

        return "MOVER"

In [7]:
agente = AgenteReactivo(entorno)

percepcion = agente.percibir()

print("Percepción:", percepcion)
print("Decisión:", agente.decidir(percepcion))

Percepción: {'posicion': (0, 0), 'sucio': False, 'obstaculo': False}
Decisión: MOVER


**Actuadores**

In [8]:
class AgenteReactivo(Agente):

    def decidir(self, percepcion):

        if percepcion["sucio"]:
            return "ASPIRAR"

        return random.choice([
            "ARRIBA",
            "ABAJO",
            "IZQUIERDA",
            "DERECHA"
        ])

    def ejecutar(self, accion):

        if accion == "ASPIRAR":

            posicion = tuple(self.entorno.robot)

            if posicion in self.entorno.suciedad:
                self.entorno.suciedad.remove(posicion)

        else:

            movimientos = {
                "ARRIBA": (-1, 0),
                "ABAJO": (1, 0),
                "IZQUIERDA": (0, -1),
                "DERECHA": (0, 1)
            }

            dx, dy = movimientos[accion]

            nueva_fila = self.entorno.robot[0] + dx
            nueva_columna = self.entorno.robot[1] + dy

            if 0 <= nueva_fila < self.entorno.filas:
                if 0 <= nueva_columna < self.entorno.columnas:

                    nueva_posicion = (
                        nueva_fila,
                        nueva_columna
                    )

                    if nueva_posicion not in self.entorno.obstaculos:
                        self.entorno.robot = list(nueva_posicion)

**Primer agente autónomo**

In [9]:
def ejecutar_agente(agente, pasos=30):

    historial = []

    for paso in range(pasos):

        percepcion = agente.percibir()

        accion = agente.decidir(percepcion)

        agente.ejecutar(accion)

        historial.append({
            "paso": paso + 1,
            "percepcion": percepcion,
            "accion": accion,
            "suciedad_restante": len(agente.entorno.suciedad)
        })

    return historial

In [10]:
entorno = Entorno()

agente = AgenteReactivo(entorno)

historial = ejecutar_agente(agente, 50)

entorno.mostrar()

⬜ 🤖 ⬜ ⬛ ⬜
⬜ ⬜ ⬜ ⬜ ⬜
⬜ ⬛ ⬜ ⬛ 🟫
⬜ 🟫 🟫 ⬜ ⬜
⬛ ⬜ 🟫 ⬜ ⬜


**Medición de rendimiento**

In [11]:
def evaluar(historial):

    suciedad_final = historial[-1]["suciedad_restante"]

    movimientos = sum(
        1 for evento in historial
        if evento["accion"] != "ASPIRAR"
    )

    limpiezas = sum(
        1 for evento in historial
        if evento["accion"] == "ASPIRAR"
    )

    puntuacion = (
        limpiezas * 10
        - movimientos
        - suciedad_final * 5
    )

    return {
        "limpiezas": limpiezas,
        "movimientos": movimientos,
        "suciedad_final": suciedad_final,
        "puntuacion": puntuacion
    }

In [12]:
resultado = evaluar(historial)

resultado

{'limpiezas': 2, 'movimientos': 48, 'suciedad_final': 4, 'puntuacion': -48}

**Experimento**

¿qué ocurre si cambiamos la métrica?

In [ ]:
# Métrica A
puntuacion = limpiezas * 10

In [ ]:
# Métrica B
puntuacion = limpiezas * 10 - movimientos

In [ ]:
# Métrica C
puntuacion = (
    limpiezas * 10
    - movimientos
    - suciedad_final * 5
)

### Reflexión

1. ¿Cuál métrica representa mejor el objetivo del agente?

2. ¿Puede un agente obtener una puntuación alta comportándose
   de una manera que realmente no queremos?

3. ¿Qué ocurre si solamente premiamos la cantidad de aspiraciones?

4. ¿Qué comportamiento induciría una penalización excesiva
   por movimiento?

5. ¿Qué otros factores podríamos incorporar?

**Agente racional**

reactivo ≠ necesariamente racional.

El agente racional debe escoger, según la información disponible, la acción que maximiza el rendimiento esperado.

In [15]:
class AgenteRacional(AgenteReactivo):

    def decidir(self, percepcion):

        if percepcion["sucio"]:
            return "ASPIRAR"

        # Evitar acciones evidentemente peligrosas
        acciones = [
            "ARRIBA",
            "ABAJO",
            "IZQUIERDA",
            "DERECHA"
        ]

        return random.choice(acciones)


**Agregar memoria**

In [16]:
class AgenteConMemoria(AgenteRacional):

    def __init__(self, entorno):
        super().__init__(entorno)
        self.memoria = []

    def percibir(self):

        percepcion = super().percibir()

        self.memoria.append(percepcion)

        return percepcion

In [17]:
entorno = Entorno()

agente = AgenteConMemoria(entorno)

historial = ejecutar_agente(agente, 20)

print("Percepciones almacenadas:")
print(len(agente.memoria))

Percepciones almacenadas:
20


**Agente BDI**

Esta sería una de las partes más interesantes del laboratorio.

El modelo BDI trabaja con:
* **BELIEFS:** ¿Qué creo?
* **DESIRES:** ¿Qué quiero lograr?
* **INTENTIONS:** ¿Qué he decidido hacer?

In [18]:
class AgenteBDI:

    def __init__(self, entorno):

        self.entorno = entorno

        self.beliefs = {}

        self.desires = [
            "MANTENER_LIMPIA_LA_HABITACION"
        ]

        self.intentions = []

    def actualizar_beliefs(self):

        posicion = tuple(self.entorno.robot)

        self.beliefs["posicion"] = posicion

        self.beliefs["sucio"] = (
            posicion in self.entorno.suciedad
        )

        self.beliefs["suciedad_restante"] = (
            len(self.entorno.suciedad)
        )

    def generar_intencion(self):

        if self.beliefs["sucio"]:
            self.intentions = ["ASPIRAR"]

        elif self.beliefs["suciedad_restante"] > 0:
            self.intentions = ["EXPLORAR"]

        else:
            self.intentions = ["FINALIZAR"]

    def mostrar_estado(self):

        print("BELIEFS:")
        print(self.beliefs)

        print("\nDESIRES:")
        print(self.desires)

        print("\nINTENTIONS:")
        print(self.intentions)

In [19]:
entorno = Entorno()

agente = AgenteBDI(entorno)

agente.actualizar_beliefs()
agente.generar_intencion()
agente.mostrar_estado()

BELIEFS:
{'posicion': (0, 0), 'sucio': False, 'suciedad_restante': 6}

DESIRES:
['MANTENER_LIMPIA_LA_HABITACION']

INTENTIONS:
['EXPLORAR']


## Experimento

Construya tres agentes:

A. AgenteReactivo
B. AgenteDeliberativo
C. AgenteHibrido

Utilice exactamente el mismo entorno.

Compare:

- Número de movimientos
- Cantidad de suciedad limpiada
- Tiempo/pasos
- Puntuación
- Capacidad de recuperación ante cambios

**PEAS**

Después pasaría de código a diseño de agentes.

El material utiliza el esquema:

Performance – Environment – Actuators – Sensors, ejemplificado con un taxista automático.

In [20]:
peas = {
    "Performance": [
        "Cantidad de suciedad eliminada",
        "Número de movimientos",
        "Energía consumida"
    ],

    "Environment": [
        "Habitación",
        "Obstáculos",
        "Suciedad"
    ],

    "Actuators": [
        "Mover arriba",
        "Mover abajo",
        "Mover izquierda",
        "Mover derecha",
        "Aspirar"
    ],

    "Sensors": [
        "Sensor de suciedad",
        "Sensor de obstáculos",
        "Sensor de posición"
    ]
}

for elemento, valores in peas.items():

    print(f"\n{elemento}")

    for valor in valores:
        print(" •", valor)


Performance
 • Cantidad de suciedad eliminada
 • Número de movimientos
 • Energía consumida

Environment
 • Habitación
 • Obstáculos
 • Suciedad

Actuators
 • Mover arriba
 • Mover abajo
 • Mover izquierda
 • Mover derecha
 • Aspirar

Sensors
 • Sensor de suciedad
 • Sensor de obstáculos
 • Sensor de posición


**Actividad**

Cada grupo de estudiantes definirá el PEAS para:

* Robot agrícola.
* Tutor inteligente.
* Vehículo autónomo.
* Sistema de diagnóstico.
* Sistema de monitoreo de red.
* Robot de inspección.

**Ejercicios**

In [ ]:
"""
Ejercicio 1 — Agente termostato

Construya un agente que perciba:
- temperatura actual;
- temperatura objetivo.

Y pueda:

- encender calefacción;
- apagar calefacción;
- encender refrigeración.

Debe entregar:

- código;
- tabla de percepciones;
- acciones;
- medida de rendimiento;
- PEAS.

"""

In [ ]:
"""
Ejercicio 2 — Agente semáforo

Construya un agente que controle un semáforo.
        a
Debe considerar:

- cantidad de vehículos;
- tiempo de espera;
- estado del semáforo.

Pregunta: ¿El agente es reactivo o proactivo? Justifique.

"""